In [1]:
from tqdm import tqdm
from architecture import Embeddings, PositionalEncoding, Encoder, Batch,BinaryClassifier, PGKPR
import torch.nn as nn
import torch
import math
from utils import loadVocab, vocab_path

In [40]:
mini_input_size = 64

ecbplus_inputs = torch.load("ecbplus_tokenized_inputs.pt")
ecbplus_mini_inputs = torch.clone(ecbplus_inputs[:mini_input_size])

ecbplus_labels = torch.load("ecbplus_labels.pt")
ecbplus_mini_labels = torch.clone(ecbplus_labels[:mini_input_size])

In [3]:
LOWERCASE = True
token2idx, idx2token = loadVocab(vocab_path, lowercase=LOWERCASE) # I should go back to the first stage and understand better how the vocab is built. However, it is not subtokened
vocab_size = len(token2idx)


In [4]:
# Encode data
#Initialize
encoder = Encoder(num_layers=6, d_model=512, nhead=8, d_ff=2048, dropout=0.1, return_layer=6)

#Manual run 
embed_test = Embeddings(512, vocab_size)
pos_encod_test = PositionalEncoding(512, 0.1, double=False)

src_embed_pos = nn.Sequential(Embeddings(512, vocab_size), #Token id to embedding
                                           PositionalEncoding(512, 0.1, double=False))

ecbplus_mini_inputs_mask = (ecbplus_mini_inputs != 0).unsqueeze(-2)

xx = src_embed_pos.forward(ecbplus_mini_inputs)
x = embed_test.forward(ecbplus_mini_inputs)
y  = pos_encod_test.forward(x)

final_encoder_layer, intermediate_encoder_layer = encoder.forward(y,ecbplus_mini_inputs_mask)

## 'Decode' Part for CDCR.

binary_classifier = BinaryClassifier(hidden_size=512, layer_norm_eps=1e-12)
output = binary_classifier(final_encoder_layer)



In [5]:
# model
#D_model is hidden_state size
D_MODEL = 512

MID_LOSS = True
DOUBLE_POSITION = False
tqdm_print_step = 1
model_save_path = 'models/'
BATCH_SIZE = 8
VAL_BATCH_SIZE = 8 

In [6]:
model = PGKPR(d_model=D_MODEL, vocab=vocab_size, mid_loss=MID_LOSS, double_pe=DOUBLE_POSITION,
                          shared_emb=True)

c:\Users\sago\Documents\GitHub\CDCR_lexical_diversity\MTL\PGKPR_adapted\architecture.py:323: UserWarning: nn.init.xavier_uniform is now deprecated in favor of nn.init.xavier_uniform_.
  nn.init.xavier_uniform(p)


In [7]:

model.eval()
model.forward(src=ecbplus_mini_inputs,src_mask=ecbplus_mini_inputs_mask)

tensor([[0.8536],
        [0.8454],
        [0.8360],
        [0.8331],
        [0.7965],
        [0.8204],
        [0.8547],
        [0.8269],
        [0.8521],
        [0.8313],
        [0.8530],
        [0.8481],
        [0.8606],
        [0.8525],
        [0.8391],
        [0.8311],
        [0.8089],
        [0.8173],
        [0.8548],
        [0.8535],
        [0.8525],
        [0.8618],
        [0.8407],
        [0.8628],
        [0.8543],
        [0.8254],
        [0.8340],
        [0.8514],
        [0.8477],
        [0.8506],
        [0.8288],
        [0.8624],
        [0.8625],
        [0.8604],
        [0.8527],
        [0.8413],
        [0.8645],
        [0.8639],
        [0.8392],
        [0.8273],
        [0.8353],
        [0.8425],
        [0.8247],
        [0.8322],
        [0.8526],
        [0.8286],
        [0.8303],
        [0.8338],
        [0.8392],
        [0.8507],
        [0.8056],
        [0.8003],
        [0.8501],
        [0.8524],
        [0.8520],
        [0

In [ ]:
# I will continue to analyze whether I need that Batch masking which seems to be more for next sentence prediction
# It is indeed like that but I like the concept of the batch doing the marking. I will replicate it 

In [8]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

In [41]:
train_data = ecbplus_mini_inputs, ecbplus_mini_labels 

val_data = ecbplus_mini_inputs, ecbplus_mini_labels #Temporarily same data to validate run

In [42]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
train_data = TensorDataset(*train_data)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=BATCH_SIZE)

val_data = TensorDataset(*val_data)
val_dataloader = DataLoader(val_data, batch_size=VAL_BATCH_SIZE) 

dataloader = {'train': train_dataloader, 'val': val_dataloader}

In [43]:
def run_once(model, batch,loss_function):
    predicted_probs= model.forward(src=batch.src,src_mask=batch.src_mask)
    predicted_probs = predicted_probs.squeeze()
    print('predicted ',predicted_probs.dtype)
    print('batch tgt ',batch.trg.dtype)
    loss = loss_function(predicted_probs, batch.trg)
    return loss

loss_function = torch.nn.BCELoss()
best_loss = float("inf")

for epoch in range(5): #TODO define epochs parameter from config file
    # scheduler.step()
    print(f'\n EPOCH {epoch}')
    model.train()
    total_loss = 0
    #cdcr_loss, pi_loss= 0, 0
    
    for phase in ['train', 'val']:
        if phase=='train':
            model.train()
        else:
            model.eval()

        with tqdm(total=len(dataloader[phase])) as pbar: 
            #Dataloader should be controlled to send a batch of one task only.
            for i, data in enumerate(dataloader[phase]):
                src, tgt= data #dataloader must have source and target, in my case the label
                batch = Batch(src, tgt)
                # src, tgt= src.cuda() tgt.cuda()#TODO send everything to device. I could adapt it to finding device first
                binary_loss = run_once(model,batch,loss_function)
                total_loss += binary_loss #It would be a good idea to monitor both losses separately
                
                if i % tqdm_print_step == 0:
                        pbar.update(tqdm_print_step)
                        pbar.set_description(
                            f'train loss: {binary_loss:.4f}')
                        
        epoch_loss = total_loss / len(dataloader[phase]) #Mean relative to dataloader size

        print(f"epoch loss: {epoch_loss}")

        if phase == 'val' and (epoch_loss < best_loss or epoch % 10 == 0) and epoch > 5:
            print("save model")
            best_loss = epoch_loss
            torch.save(model.state_dict(), f'{model_save_path}/epoch{epoch}_loss_{epoch_loss}.pkl')



 EPOCH 0


train loss: 1.8707:  12%|█▎        | 1/8 [00:01<00:11,  1.58s/it]

predicted  torch.float32
batch tgt  torch.float32


train loss: 1.2467:  25%|██▌       | 2/8 [00:03<00:09,  1.61s/it]

predicted  torch.float32
batch tgt  torch.float32


...

predicted  torch.float32
batch tgt  torch.float32


train loss: 1.2400:  25%|██▌       | 2/8 [00:04<00:13,  2.17s/it]

predicted  torch.float32
batch tgt  torch.float32


train loss: 1.2400:  25%|██▌       | 2/8 [00:07<00:21,  3.57s/it]


RuntimeError: [enforce fail at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10\core\impl\alloc_cpu.cpp:72] data. DefaultCPUAllocator: not enough memory: you tried to allocate 10240000 bytes.

In [ ]:
# Types seem to be fine, now I have to add the the optimizer step.